In [8]:
# Assignment 01
from pyspark.sql import SparkSession
import json

 
spark = SparkSession.builder \
    .appName("Generate Parquet Data") \
    .config("spark.sql.parquet.enableStats", "true") \
    .getOrCreate()

dict = {"num_rows": "", "format": "", "status": ""}

df = spark.read.parquet("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/FirstProject/src/leetcode/large_practice_data.parquet")

df.show(5)

if df is not None:
    dict["num_rows"] = df.count()
    
    # Get the source path from the Spark plan
    logical_plan = df._jdf.queryExecution().logical()
        
    source_info = logical_plan.toString()

    if "Parquet" in source_info:
        file_format = "parquet"
    elif "CSV" in source_info:
        file_format = "csv"
    else:
        file_format = "unknown"
    print(f"Detected Format: {file_format}")
    dict["format"] = file_format

    dict["status"] = "success"



print (dict)

#create a json file to store the metadata

with open("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/Feb_2026/metadata.json", "w") as f:
    json.dump(dict, f, indent=4)


 
spark.stop()

+---+--------------+-------+-----------+--------------------+-----+----------------+------------------+------------------+-----------+-----------------+---------------+---------------+---------------+
| id|transaction_id|user_id|   category|           timestamp| name|           email|            amount|             score|category_id|         discount|date_mm_dd_yyyy|date_dd_mm_yyyy|date_yyyy_mm_dd|
+---+--------------+-------+-----------+--------------------+-----+----------------+------------------+------------------+-----------+-----------------+---------------+---------------+---------------+
|900|           900|   4853|       null|2023-02-13 15:35:...|Alice|            null| 376.2585170603643|              null|       null|             null|     02-13-2023|           null|           null|
|901|           901|   5779|Electronics|2023-08-25 04:44:...|Alice|            null|105.43840119496156| 42.47771204426548|          8|             null|     08-25-2023|     25-08-2023|           n

In [ ]:
# Assignment 02
''' This should read the record.json file along with input file .
    If the status in record.json is valid then perform the below activity else exit the program with exception as "Data is not correct" 
    The Names should be capitalised .
    The date should be uniform ( DD-MM-YYYY)
    Null should be replaced with Unknown for text field , 0 for numeric fields.'''

import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, IntegerType, FloatType, DoubleType,
    DateType, TimestampType, BooleanType
)


spark = SparkSession.builder.appName("Read Parquet Data").getOrCreate()


with open("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/Feb_2026/metadata.json", "r") as f:
    metadata = json.load(f)

if metadata["status"] == "success":
    
    df = spark.read.parquet(
        "/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/FirstProject/src/leetcode/large_practice_data.parquet"
    )
    df.show(10, truncate=False)
    
    for field in df.schema.fields:
        col_name = field.name
        dtype = field.dataType

       
        if isinstance(dtype, StringType):
            df = df.withColumn(col_name, F.initcap(F.col(col_name)))
        elif isinstance(dtype, (DateType, TimestampType)):
            df = df.withColumn(col_name, F.date_format(F.col(col_name), "dd-MM-yyyy"))

    for field in df.schema.fields:
        col_name = field.name
        dtype = field.dataType

        if isinstance(dtype, StringType):
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit("Unknown")))

        elif isinstance(dtype, (IntegerType, FloatType, DoubleType)):
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit(0)))

        elif isinstance(dtype, BooleanType):
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit(False)))

  
    df.show(10, truncate=False)
else:
    raise Exception("Data is not correct")


spark.stop()


'''
            if "date" in col_name.lower():
                df = df.withColumn(
                    col_name,
                    F.date_format(
                        F.coalesce(
                            F.to_date(F.col(col_name), "MM-dd-yyyy"),
                            F.to_date(F.col(col_name), "dd-MM-yyyy"),
                            F.to_date(F.col(col_name), "yyyy-MM-dd")
                        ),
                        "dd-MM-yyyy"
                    )
                )
            elif "name" in col_name.lower():'''
       

+---+--------------+-------+-----------+--------------------------+-----+----------------+------------------+------------------+-----------+------------------+---------------+---------------+---------------+
|id |transaction_id|user_id|category   |timestamp                 |name |email           |amount            |score             |category_id|discount          |date_mm_dd_yyyy|date_dd_mm_yyyy|date_yyyy_mm_dd|
+---+--------------+-------+-----------+--------------------------+-----+----------------+------------------+------------------+-----------+------------------+---------------+---------------+---------------+
|900|900           |4853   |null       |2023-02-13 15:35:59.919691|Alice|null            |376.2585170603643 |null              |null       |null              |02-13-2023     |null           |null           |
|901|901           |5779   |Electronics|2023-08-25 04:44:40.928345|Alice|null            |105.43840119496156|42.47771204426548 |8          |null              |08-25-202

'\n            if "date" in col_name.lower():\n                df = df.withColumn(\n                    col_name,\n                    F.date_format(\n                        F.coalesce(\n                            F.to_date(F.col(col_name), "MM-dd-yyyy"),\n                            F.to_date(F.col(col_name), "dd-MM-yyyy"),\n                            F.to_date(F.col(col_name), "yyyy-MM-dd")\n                        ),\n                        "dd-MM-yyyy"\n                    )\n                )\n            elif "name" in col_name.lower():'

In [29]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Trasnformations and aggregations using above data").getOrCreate()

df1 = spark.read.parquet(
    "/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/FirstProject/src/leetcode/large_practice_data.parquet"
)

df1.show(10, truncate=False)
df1.printSchema()

df1 = df1.fillna({"name": "Unknown", "amount": 0, "category": "default_category"})

df1.withColumn("name", F.upper(F.col("name"))).write.mode("overwrite").parquet("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/Feb_2026/processed_data1.parquet")

df_processed_date = spark.read.parquet("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/Feb_2026/processed_data1.parquet")
print("showing processed data with name in upper case")
df_processed_date.show(10, truncate=False)

# amount > 0 
df2 = df1.filter(F.col("amount") > 0)

df2.show(10, truncate=False)


df3 = df2.withColumn("name", F.upper(F.col("name")))

print("showing df3 data after change name to upper case")

df3.select("name").show(10, truncate=False)

df4 = df3.groupby(F.col("category")).agg(F.avg("amount").alias("average_amount"))

df4.show(10, truncate=False)
    
df5 = df3.groupby(F.col("category")).agg(F.countDistinct(F.col("email")).alias("unique_emails"), F.sum("amount").alias("total_amount"))

df5.show(10, truncate=False)

df3.groupby(F.col("category")).agg(F.count("email").alias("email_count")).show(10, truncate=False)

df3.select("email").show(10, truncate=False)

df3.dropDuplicates(["email"]).show(10, truncate=False)

df6 = df3.select([F.col(c) for c in df3.columns])

print("Df6 Data:")
df6.show(10, truncate=False)

df7 = df3.filter((F.col("category").isNotNull()) & (F.col("category") != "default_category"))

print("showing not null categories in df7")
df7.show(10, truncate=False)

#update Electronics to Electronics_items in category column
df8 = df3.withColumn("category", F.when(F.col("category") == "Electronics", "Electronic_items").otherwise("default_category"))
print("showing updated category in df8")
df8.show(10, truncate=False)

spark.stop()

+---+--------------+-------+-----------+--------------------------+-----+----------------+------------------+------------------+-----------+------------------+---------------+---------------+---------------+
|id |transaction_id|user_id|category   |timestamp                 |name |email           |amount            |score             |category_id|discount          |date_mm_dd_yyyy|date_dd_mm_yyyy|date_yyyy_mm_dd|
+---+--------------+-------+-----------+--------------------------+-----+----------------+------------------+------------------+-----------+------------------+---------------+---------------+---------------+
|900|900           |4853   |null       |2023-02-13 15:35:59.919691|Alice|null            |376.2585170603643 |null              |null       |null              |02-13-2023     |null           |null           |
|901|901           |5779   |Electronics|2023-08-25 04:44:40.928345|Alice|null            |105.43840119496156|42.47771204426548 |8          |null              |08-25-202

26/02/20 16:02:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/20 16:02:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/20 16:02:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/02/20 16:02:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/20 16:02:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


+---+--------------+-------+----------------+--------------------------+-----+----------------+------------------+-------------------+-----------+------------------+---------------+---------------+---------------+
|id |transaction_id|user_id|category        |timestamp                 |name |email           |amount            |score              |category_id|discount          |date_mm_dd_yyyy|date_dd_mm_yyyy|date_yyyy_mm_dd|
+---+--------------+-------+----------------+--------------------------+-----+----------------+------------------+-------------------+-----------+------------------+---------------+---------------+---------------+
|0  |0             |34958  |Electronics     |2023-09-14 03:36:12.058573|BOB  |null            |0.0               |null               |2          |8.436901938288576 |null           |14-09-2023     |null           |
|1  |1             |84804  |default_category|2023-11-25 18:44:39.825281|ALICE|null            |0.0               |null               |null      

In [2]:
from pyspark.sql import SparkSession

# Stop any existing Spark session
try:
    spark.stop()
except:
    pass

# Create SparkSession with MySQL connector auto-downloaded
spark = SparkSession.builder \
    .appName("mysql connection") \
    .config("spark.jars.packages", "mysql:mysql-connector-java:8.0.33") \
    .getOrCreate()

# MySQL connection parameters
HOST = "3.14.82.61"
USER = "root"
PASSWORD = "Yashwant!14"
DATABASE = "pipeline_assignment_db"
TABLE = "processed_parquet_data"

jdbc_url = f"jdbc:mysql://{HOST}:3306/{DATABASE}?useSSL=false&allowPublicKeyRetrieval=true"

df_processed_data = spark.read.parquet("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/Feb_2026/processed_data1.parquet")

df_processed_data.write.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE) \
    .option("user", USER) \
    .option("password", PASSWORD) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .mode("overwrite") \
    .save()
# Read table from MySQL
df = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE) \
    .option("user", USER) \
    .option("password", PASSWORD) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

print("✅ Connection successful, here are some rows:")
df.show()

✅ Connection successful, here are some rows:


+---+--------------+-------+----------------+-------------------+-----+----------------+------------------+-----------------+-----------+------------------+---------------+---------------+---------------+
| id|transaction_id|user_id|        category|          timestamp| name|           email|            amount|            score|category_id|          discount|date_mm_dd_yyyy|date_dd_mm_yyyy|date_yyyy_mm_dd|
+---+--------------+-------+----------------+-------------------+-----+----------------+------------------+-----------------+-----------+------------------+---------------+---------------+---------------+
|300|           300|   6195|default_category|2023-03-13 14:29:47|ALICE|            null|21.876792791262922|             null|       null|              null|     03-13-2023|           null|           null|
|600|           600|  94902|default_category|2023-03-15 21:53:14|ALICE|            null|               0.0|91.39744024171783|          2|              null|     03-15-2023|     15-